# لبِق — تدريب Saudi LoRA v2 (بعد اعتماد بيانات المرحلة الثانية)

**الغرض:** تدريب Qwen3.5-4B باستخدام ملف التدريب المعتمد، ثم مقارنة النموذج الأساسي مع الناتج على مواقف مستقلة.

- بيانات التدريب: 958 مثالًا، منها 178 تم تحديثها بناء على المراجعة المعتمدة، و780 مثالًا أصليًا لم تخضع للمراجعة نفسها.
- حالتان في ملف الاعتماد **مستبعدتان من التدريب** احترازيًا بسبب تضارب محتمل في هدف الرفض.
- التحقق: 114 مثالًا؛ الاختبار: 114؛ المقارنة السريعة: 38 موقفًا مختلفًا من Test.
- النوتبوك **لم يُشغَّل على GPU عند إنشاء الحزمة**؛ تحقق النتائج بعد تشغيله بنفسك.

**المطلوب:** Google Colab مع L4 أو A100 GPU، وملف `LABEQ_Approved_Training_Data.zip` المرفق بالحزمة. افتحي هذا النوتبوك في جلسة تدريب جديدة، لا داخل جلسة API الحية.

## 1) ارفعي ملف البيانات المعتمدة
لا ترفعين حزمة الـ12 ألف الأصلية هنا؛ ارفعي فقط `LABEQ_Approved_Training_Data.zip`.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, io, json
uploads=files.upload()
if 'LABEQ_Approved_Training_Data.zip' not in uploads:
    raise FileNotFoundError('اختاري LABEQ_Approved_Training_Data.zip فقط')
root=Path('/content/LABEQ_Approved_Training')
root.mkdir(exist_ok=True,parents=True)
with zipfile.ZipFile(io.BytesIO(uploads['LABEQ_Approved_Training_Data.zip'])) as z:
    for info in z.infolist():
        p=Path(info.filename)
        if p.is_absolute() or '..' in p.parts: raise ValueError('Unsafe archive path')
    z.extractall(root)
DATA=root/'data'
for fn in ['train_full_958.jsonl','train_gold_178.jsonl','validation_114.jsonl','test_38_one_per_scenario.jsonl']:
    assert (DATA/fn).exists(),fn
manifest=json.loads((DATA/'dataset_manifest.json').read_text(encoding='utf-8'))
print('Data summary:',manifest['counts'])

## 2) ثبّتي المكتبات
Qwen3.5 يحتاج Transformers v5. نستخدم LoRA بدقة 16-bit، **مو QLoRA بدقة 4-bit**. ملاحظة: إذا ظهر طلب لإعادة تشغيل Runtime بعد التثبيت، أعيدي تشغيله ونفذي الخلايا من البداية.

In [ ]:
%pip install -q -U unsloth unsloth_zoo datasets trl peft "transformers>=5.0" "torchao>=0.16.0" safetensors

## 3) قراءة مجموعة التدريب وتحضير Qwen
المجموعة الأساسية الافتراضية هي `train_full_958.jsonl`؛ يمكن إجراء تجربة إضافية باستخدام `train_gold_178.jsonl` ولكن حجمها صغير وقد يسبب Overfitting.

In [ ]:
import torch, inspect, json
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
assert torch.cuda.is_available(), 'اختاري Runtime > Change runtime type > GPU'
print('GPU:',torch.cuda.get_device_name(0))

def load_prompt_completion(file):
    arr=[json.loads(line) for line in open(file,encoding='utf-8') if line.strip()]
    assert all(x['prompt'][-1]['role']=='user' and x['completion'][0]['role']=='assistant' for x in arr)
    return Dataset.from_list(arr)

TRAIN_FILE='train_full_958.jsonl'  # الافتراضي 958 مثالًا
train_ds=load_prompt_completion(DATA/TRAIN_FILE)
valid_ds=load_prompt_completion(DATA/'validation_114.jsonl')
print('Train:',len(train_ds),'Validation:',len(valid_ds))
MODEL_ID='unsloth/Qwen3.5-4B'
MAX_LENGTH=768
model,tokenizer=FastLanguageModel.from_pretrained(
    model_name=MODEL_ID, max_seq_length=MAX_LENGTH,
    load_in_4bit=False, load_in_16bit=True, full_finetuning=False)
model=FastLanguageModel.get_peft_model(
    model,r=16,lora_alpha=16,lora_dropout=0,bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth',random_state=3407,max_seq_length=MAX_LENGTH)
print('✅ Qwen3.5-4B مع LoRA جاهز')

## 4) Train — مع تقييم دوري
هنا الاختلاف عن Pilot السابق: بدل 80 خطوة (~0.33 Epoch)، نجرّب دورتين كاملتين مع فحص Validation. **عدد الدورات إعداد تجريبي** وليس إثباتًا أنها الأفضل. احتفظي بسجل Validation Loss وشوفي جودة المخرجات قبل اعتماد الأوزان.

In [ ]:
params=inspect.signature(SFTConfig).parameters
kwargs={
    'output_dir':'/content/LABEQ_Saudi_LoRA_v2_runs',
    'per_device_train_batch_size':1,
    'gradient_accumulation_steps':4,
    'num_train_epochs':2,
    'learning_rate':5e-5,
    'warmup_ratio':0.05,
    'logging_steps':20,
    'save_steps':100,
    'eval_steps':100,
    'save_strategy':'steps',
    'optim':'adamw_8bit',
    'report_to':'none',
    'seed':3407,
    'bf16':torch.cuda.is_bf16_supported(),
    'fp16':not torch.cuda.is_bf16_supported(),
    'gradient_checkpointing':True,
}
if 'eval_strategy' in params: kwargs['eval_strategy']='steps'
elif 'evaluation_strategy' in params: kwargs['evaluation_strategy']='steps'
if 'max_length' in params:kwargs['max_length']=MAX_LENGTH
else:kwargs['max_seq_length']=MAX_LENGTH
if 'completion_only_loss' in params: kwargs['completion_only_loss']=True
cfg=SFTConfig(**{k:v for k,v in kwargs.items() if k in params})
trainer_params=inspect.signature(SFTTrainer).parameters
processor_kw={'processing_class':tokenizer} if 'processing_class' in trainer_params else {'tokenizer':tokenizer}
trainer=SFTTrainer(model=model,args=cfg,train_dataset=train_ds,eval_dataset=valid_ds,**processor_kw)
train_result=trainer.train()
print('✅ انتهى التدريب. Train metrics:',train_result.metrics)

## 5) مقارنة منصفة: Qwen الأساسي مقابل LoRA الجديد
نفس الرسائل، نفس عدد التوكنات ونفس إعدادات التوليد بدون Sampling. النتائج تحفظ في CSV **بدون تقييم آلي لجودة اللهجة**. الهدف من الملف مراجعة بشرية: المعنى / اللهجة / ملاءمة العلاقة / اختلاف الأساليب.

In [ ]:
from contextlib import nullcontext
import csv
FastLanguageModel.for_inference(model)
model.eval()
test=[json.loads(line) for line in open(DATA/'test_38_one_per_scenario.jsonl',encoding='utf-8') if line.strip()]
assert len(test)==38
if not hasattr(model,'disable_adapter'):
    raise RuntimeError('تعذر تعطيل LoRA لهذا النموذج؛ لا تسمّيه اختبار Base حتى نصلح الطريقة')

def reply_for(messages, use_adapter=True):
    prompt=tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True,enable_thinking=False)
    token_batch=tokenizer(prompt,return_tensors='pt',add_special_tokens=False).to(model.device)
    disable_context=nullcontext() if use_adapter else model.disable_adapter()
    with disable_context, torch.inference_mode():
        out=model.generate(**token_batch,max_new_tokens=160,do_sample=False,
                           eos_token_id=tokenizer.eos_token_id,
                           pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
    return tokenizer.decode(out[0,token_batch['input_ids'].shape[-1]:],skip_special_tokens=True).strip()

responses=[]
for n,item in enumerate(test,1):
    source=item['prompt']; reference=item['completion'][0]['content']
    base=reply_for(source,use_adapter=False)
    tuned=reply_for(source,use_adapter=True)
    responses.append({'case':n,'prompt':source[1]['content'],'reference_hidden_do_not_use_for_training':reference,
                      'qwen_base':base,'qwen_new_lora':tuned,
                      'human_score_meaning_1to5':'','human_score_saudi_1to5':'','human_score_social_1to5':'',
                      'human_winner_base_or_lora_or_tie':''})
    print(f'{n}/38', 'BASE:',base[:110], '\nLoRA:',tuned[:110], '\n')
path='/content/LABEQ_Base_vs_New_LoRA_38.csv'
with open(path,'w',encoding='utf-8-sig',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(responses[0]));w.writeheader();w.writerows(responses)
print('✅ Saved:',path)

## 6) حفظ LoRA الجديد وتنزيل الملفات
لا تستبدلين أوزان V16 القديمة إلا **بعد** مراجعة نتائج المقارنة؛ مجرد انتهاء التدريب لا يثبت أن LoRA الجديد أفضل.

In [ ]:
from pathlib import Path
from google.colab import files
import shutil, json, time
save_dir=Path('/content/LABEQ_Saudi_LoRA_v2')
trainer.model.save_pretrained(str(save_dir))
tokenizer.save_pretrained(str(save_dir))
assert (save_dir/'adapter_config.json').exists()
assert list(save_dir.glob('adapter_model*.safetensors'))
archive=shutil.make_archive(str(save_dir),'zip',root_dir=str(save_dir))
print('Saved adapter:',archive)
files.download(archive)
files.download('/content/LABEQ_Base_vs_New_LoRA_38.csv')

## 7) ربط النتيجة بواجهة لبِق V16

1. احتفظي بـ`LABEQ_Saudi_LoRA_v2.zip` ولا تحذفي LoRA القديم.
2. في **جلسة Colab جديدة للتشغيل** افتحي `LABEQ_Colab_API_Connect_V16.ipynb` من مجلد التطبيق.
3. عند خلية رفع LoRA اختاري ZIP الجديد؛ قد تطلب الخلية اسمًا قديمًا لكنها تقبل أي ZIP يحتوي `adapter_config.json` و`adapter_model.safetensors`.
4. كملي تحميل فيصل وتشغيل API وCloudflare، وحدّثي `.env` في موقع V16 عبر `python3 configure_model.py`.
5. افحصي `/api/model-health`، ثم اختبري أن النتيجة تظهر `router_source=faisal` و`reply_source=qwen_lora_generated` في طلب فعلي.

⚠️ **لم يتم التشغيل أو الدمج تلقائيًا عند إنشاء هذه الحزمة.** صور الموقع القديمة ليست دليلًا أن هذه الأوزان هي التي تولّد الردود.